# Triagem com BERTimbau (Google Colab)

Ajuste fino do `neuralmind/bert-base-portuguese-cased` no conjunto prata.

**Antes de rodar:**
1. No PC: `python scripts/export_for_colab.py`.
2. No Colab: **Ambiente de execução → Alterar o tipo → GPU T4**.
3. Envie para o Colab (ícone de pasta à esquerda → upload) os 3 arquivos de `data/processed/colab/`.

**Saída:** `triage_preds_val_bertimbau.csv` e `triage_preds_test_bertimbau.csv`. Baixe e coloque em `eval/results/` no PC; a avaliação é feita lá, com os rótulos humanos que nunca vêm para o Colab.

A latência é medida **em CPU**, um ticket por vez, para comparar de forma justa com as outras abordagens.

In [ ]:
!pip -q install transformers==5.22.1 accelerate

In [ ]:
import time

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
MODEL = "neuralmind/bert-base-portuguese-cased"
MAX_LEN = 128
set_seed(SEED)
print("GPU:", torch.cuda.is_available())

In [ ]:
train = pd.read_csv("silver_train.csv")
labels = sorted(train["label"].unique())
label2id = {lab: i for i, lab in enumerate(labels)}
id2label = {i: lab for lab, i in label2id.items()}

# 10% do prata para acompanhar o treino (não é avaliação final).
holdout = train.sample(frac=0.1, random_state=SEED)
fit = train.drop(holdout.index)
tokenizer = AutoTokenizer.from_pretrained(MODEL)


class Texts(Dataset):
    def __init__(self, frame):
        self.enc = tokenizer(frame["text"].tolist(), truncation=True, max_length=MAX_LEN)
        self.labels = [label2id[x] for x in frame["label"]] if "label" in frame else None

    def __len__(self):
        return len(self.enc["input_ids"])

    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k, v in self.enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[i])
        return item


print(len(fit), "treino |", len(holdout), "acompanhamento |", len(labels), "categorias")

In [ ]:
from sklearn.metrics import f1_score

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=len(labels), id2label=id2label, label2id=label2id
)


def metrics(pred):
    y = pred.label_ids
    p = pred.predictions.argmax(-1)
    return {"macro_f1": f1_score(y, p, average="macro"), "accuracy": (y == p).mean()}


args = TrainingArguments(
    output_dir="bertimbau",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    seed=SEED,
    report_to=[],
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=Texts(fit),
    eval_dataset=Texts(holdout),
    processing_class=tokenizer,
    compute_metrics=metrics,
)
trainer.train()
print(trainer.evaluate())

In [ ]:
def predict_cpu(frame):
    """Um ticket por vez, em CPU com 2 threads: latência comparável à da API."""
    cpu_model = trainer.model.to("cpu").eval()
    torch.set_num_threads(2)
    rows = []
    for row in frame.itertuples():
        start = time.perf_counter()
        enc = tokenizer(row.text, truncation=True, max_length=MAX_LEN, return_tensors="pt")
        with torch.no_grad():
            proba = torch.softmax(cpu_model(**enc).logits, dim=-1)[0].numpy()
        best = int(np.argmax(proba))
        rows.append(
            {
                "complaint_id": row.complaint_id,
                "pred": id2label[best],
                "confidence": float(proba[best]),
                "latency_ms": (time.perf_counter() - start) * 1000,
                "input_tokens": 0,
                "output_tokens": 0,
            }
        )
    return pd.DataFrame(rows)


for split in ("val", "test"):
    preds = predict_cpu(pd.read_csv(f"gold_{split}.csv"))
    preds.to_csv(f"triage_preds_{split}_bertimbau.csv", index=False)
    print(split, len(preds), "previsões; p50", round(preds["latency_ms"].median()), "ms")

In [ ]:
from google.colab import files

files.download("triage_preds_val_bertimbau.csv")
files.download("triage_preds_test_bertimbau.csv")